In [175]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import EarlyStopping
import random
import nltk
from nltk.stem import WordNetLemmatizer
import json
import pickle

In [176]:
lemmatizer = WordNetLemmatizer()

In [177]:
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to C:\Users\ROHIT
[nltk_data]     RANE\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\ROHIT
[nltk_data]     RANE\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [178]:
words = []
classes = []
documents = []
ignore_letters = ['!', '?', ',', '.']

In [179]:
intents_file = 'intents.json'
with open(intents_file, 'r') as f:
    intents = json.load(f)

In [180]:
intents_file

'intents.json'

In [181]:
for intent in intents['intents']:
    for pattern in intent['patterns']:
        word = nltk.word_tokenize(pattern)
        words.extend(word)
        documents.append((word, intent['tag']))
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

In [182]:
words = [lemmatizer.lemmatize(w.lower()) for w in words if w not in ignore_letters]

In [183]:
words = sorted(list(set(words)))

In [184]:
classed = sorted(list(set(classes)))

In [185]:
print(len(documents), "documents")
print(len(classes), "classes", classes)
print(len(words), "unique lemmatized words", words)

47 documents
9 classes ['greeting', 'goodbye', 'thanks', 'options', 'adverse_drug', 'blood_pressure', 'blood_pressure_search', 'pharmacy_search', 'hospital_search']
87 unique lemmatized words ["'s", 'a', 'adverse', 'all', 'anyone', 'are', 'awesome', 'be', 'behavior', 'blood', 'by', 'bye', 'can', 'causing', 'chatting', 'check', 'could', 'data', 'day', 'detail', 'do', 'dont', 'drug', 'entry', 'find', 'for', 'give', 'good', 'goodbye', 'have', 'hello', 'help', 'helpful', 'helping', 'hey', 'hi', 'history', 'hola', 'hospital', 'how', 'i', 'id', 'is', 'later', 'list', 'load', 'locate', 'log', 'looking', 'lookup', 'management', 'me', 'module', 'nearby', 'next', 'nice', 'of', 'offered', 'open', 'patient', 'pharmacy', 'pressure', 'provide', 'reaction', 'related', 'result', 'search', 'searching', 'see', 'show', 'suitable', 'support', 'task', 'thank', 'thanks', 'that', 'there', 'till', 'time', 'to', 'transfer', 'up', 'want', 'what', 'which', 'with', 'you']


In [186]:
pickle.dump(words,open('words.pkl','wb'))
pickle.dump(classes,open('classes.pkl','wb'))

In [187]:
training_data = []
output_empty = [0] * len(classes)

In [188]:
for doc in documents:
    bag = []
    pattern_words = doc[0]
    pattern_words = [lemmatizer.lemmatize(word.lower()) for word in pattern_words]
    for word in words:
        bag.append(1) if word in pattern_words else bag.append(0)
        
    output_row = list(output_empty)
    output_row[classes.index(doc[1])] = 1
    
    training_data.append([bag, output_row])

In [189]:
random.shuffle(training_data)

In [190]:
x_train = np.array([x[0] for x in training_data], dtype=np.float32)
y_train = np.array([x[1] for x in training_data], dtype=np.float32)

print(x_train.shape)
print(y_train.shape)

(47, 87)
(47, 9)


In [191]:
model = Sequential()
model.add(Dense(128, input_shape = (len(train_x[0]), ), activation = 'relu' ))
#model.add(Dropout(0.5))

model.add(Dense(64, activation = 'relu'))
#model.add(Dropout(0.5))

model.add(Dense(len(train_y[0]), activation = 'softmax'))

In [192]:
sgd = SGD(learning_rate = 0.01, momentum = 0.9, nesterov = True)
model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

In [193]:
callback = tf.keras.callbacks.EarlyStopping(
    monitor='accuracy',
    min_delta=0,
    patience=20,
    verbose=1,
    mode='auto',
    baseline=None,
    restore_best_weights=False,
    start_from_epoch=0
)

In [194]:
history = model.fit(x_train, y_train, epochs = 200, verbose = 1, batch_size = 5, callbacks = callback)

Epoch 1/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.0638 - loss: 2.1931
Epoch 2/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5532 - loss: 2.0526
Epoch 3/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7447 - loss: 1.9328 
Epoch 4/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8936 - loss: 1.7951 
Epoch 5/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8936 - loss: 1.6452 
Epoch 6/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9149 - loss: 1.4777
Epoch 7/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9362 - loss: 1.2962 
Epoch 8/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9362 - loss: 1.1093  
Epoch 9/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9574 - loss: 0.9310 
Epoch 10/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9574 - loss: 0.7647
Epoch 11/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9787 - loss: 0.6252  
Epoch 12/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/ste

In [196]:
model.save('chatbot_model.h5', history)
print("Model Saved!!")

Model Saved!!
